## TP2 : Préparation et qualité des sources

### objectif :
Produire une source préparée, contrôlée et traçable pour le futur modèle décisionnel

In [1]:
import pandas as pd 
import numpy as np
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/xxxxxcn/uci-online-retail-original-workbook/Online Retail.xlsx
/kaggle/input/datasets/xxxxxcn/uci-online-retail-original-workbook/README.md


In [2]:
df= pd.read_excel('/kaggle/input/datasets/xxxxxcn/uci-online-retail-original-workbook/Online Retail.xlsx')

 **Le FactVente MVP est la première version de la table de ventes, avec un périmètre volontairement limité :**


> seulement les ventes de produits (pas les annulations/retours),
 
> seulement les lignes avec Quantity > 0 et UnitPrice > 0,

> seulement les lignes classées PRODUIT_COMMERCIAL.


### Question de départ : 
##### Comment produire une source fiable pour le futur FactVente sans supprimer abusivement des informations métier ?


On produit une source fiable en mesurant avant de toucher, en classant au lieu de supprimer, et en ne supprimant qu’une seule chose : les doublons exacts. Tout le reste est soit conservé (avec un drapeau), soit exclu de FactVente mais gardé en audit.

La distinction centrale est entre :


> * supprimée de la source : la ligne n’existe plus nulle part (réservé aux doublons techniques exacts)
> * exclue de FactVente : la ligne reste en staging/audit, tracée, avec la raison de l’exclusion.


# Raisonnement :

   **1. Quelles anomalies mesurer avant de modifier les données ?**


* Dimensions,
*  types,
*  valeurs manquantes par colonne (surtout CustomerID et Description), doublons exacts,
*  Quantity ≤ 0,
*  UnitPrice ≤ 0,
*  InvoiceNo commençant par « C »,
*  codes StockCode non commerciaux.
 
On mesure d’abord pour avoir une référence et pouvoir prouver l’effet des transformations.

**2. Pourquoi InvoiceNo + StockCode n’est pas une clé de ligne ?**

Une même facture peut contenir le même produit sur plusieurs lignes (prix différent, quantités séparées, ajouts successifs). 
Deux lignes avec le même couple peuvent donc être légitimes. 
Le dédoublonnage se fait uniquement sur l’identité des 8 colonnes.

**3. Comment distinguer un code non commercial d’un code commercial alphanumérique ?**

Une lettre dans le StockCode ne suffit pas, car des produits normaux ont des codes comme « 85123A ». On utilise la référence de classification fournie : StockCode + Description + comportement (quantités, prix). Exemples typiques de codes non commerciaux : POST, DOT, M,...etc. Les cas douteux sont marqués A_VERIFIER et documentés.


**4. Pourquoi InvoiceNo commençant par C ne doit-il pas êtresupprimé sans trace ?**

Les InvoiceNo commençant par « C » correspondent à des annulations ou retours, qui sont des événements métier réels et non des erreurs de saisie. Ils sont exclus du périmètre FactVente MVP (CA de ventes hors annulations/retours), mais conservés en staging/audit avec leur motif d’exclusion. Cela garantit la traçabilité, permet de réconcilier les volumes avant/après et laisse la possibilité d’analyser les retours ultérieurement.


**5. Que faites-vous d’un CustomerID manquant ?**

Les lignes dont le CustomerID est manquant sont conservées dans la couche préparée. Elles représentent des ventes réelles, complètes en quantité, prix, date et pays, et leur suppression fausserait le chiffre d’affaires. Aucune valeur n’est inventée : ces lignes seront rattachées à un membre « Client inconnu » lors de la modélisation (TP3).


**6. Quels contrôles permettront de prouver l’effet de vostransformations ?**

Les transformations sont prouvées par un tableau de contrôles avant/après :


* nombre de lignes,
* doublons exacts,
* CustomerID manquants,
* annulations,
* UnitPrice et Quantity non positifs,
* lignes non commerciales et somme de MontantLigne.

Chaque écart est expliqué par une règle, et une réconciliation vérifie que lignes source = doublons supprimés + lignes d’audit + lignes préparées.
Les valeurs manquantes sont étiquetées et conservées, jamais supprimées ni inventées.

# CODE / MANIPULATION

### 1. PROFILAGE ET CONTRÔLES

In [3]:
print("Dimensions :", df.shape)
print(df.dtypes)
print("\nValeurs manquantes :")
print(df.isna().sum())
print("\nDoublons exacts :", df.duplicated().sum())
print("\nQuantity :")
print(df["Quantity"].describe())
print("\nUnitPrice :")
print(df["UnitPrice"].describe())
print("\nInvoiceNo commençant par C :")
print(df["InvoiceNo"].astype(str).str.startswith("C").sum())

Dimensions : (541909, 8)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Valeurs manquantes :
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Doublons exacts : 5268

Quantity :
count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

UnitPrice :
count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

InvoiceNo commençant par C :


**Que constatez-vous ?**

La source contient: 541 909 lignes et 8 colonnes.
- 1 454 Description manquantes,
- 135 080 CustomerID manquants (≈ 24,9 %),
- 5 268 doublons exacts
- 9 288 InvoiceNo commençant par « C ».
- Quantity va de -80 995 à 80 995
- UnitPrice de -11 062,06 à 38 970,
  > ce qui révèle des retours/annulations, des ajustements et des valeurs extrêmes à examiner. Ces valeurs constituent l’état initial de référence avant toute transformation.

**Quel traitement décidez-vous d’appliquer ?**


* *Doublons exacts (5 268)* : supprimés, une copie conservée, sur les huit colonnes (suppression tracée pour la justification).
* *InvoiceNo « C » (9 288)* : événements métier réels, exclus de FactVente MVP mais conservés en audit.
* *Quantity ≤ 0 et UnitPrice ≤ 0*: classés puis exclus du périmètre (règle MVP : Quantity > 0 et UnitPrice > 0), conservés en audit avec motif.
* *CustomerID manquant (135 080)* : conservé pour rattachement ultérieur à « Client inconnu ». Pas de suppression ni d’invention de valeur.
* *Description manquante (1 454)* : étiquetée à revoir, non supprimée ;la classification se fait avec StockCode et le comportement de la ligne.
* *Valeurs extrêmes* : examinées individuellement, classées selon la référence de classification (A_VERIFIER en cas de doute), jamais supprimées automatiquement.


## **Etat initiale**

In [4]:

n = len(df)

tableau_initial = pd.DataFrame({
    "Contrôle": [
        "Nombre de lignes", "Nombre de colonnes", "Doublons exacts (8 colonnes)",
        "Quantity min", "Quantity max", "UnitPrice min", "UnitPrice max",
        "Quantity <= 0", "UnitPrice <= 0", "InvoiceNo commençant par C",
        "CustomerID manquants", "Description manquantes",
    ],
    "Avant": [
        n, df.shape[1], int(df.duplicated().sum()),
        df["Quantity"].min(), df["Quantity"].max(),
        df["UnitPrice"].min(), df["UnitPrice"].max(),
        int((df["Quantity"] <= 0).sum()), int((df["UnitPrice"] <= 0).sum()),
        int(df["InvoiceNo"].astype(str).str.startswith("C").sum()),
        int(df["CustomerID"].isna().sum()), int(df["Description"].isna().sum()),
    ],
})
tableau_initial["% des lignes"] = (tableau_initial["Avant"] / n * 100).round(2)

# Le % n'a de sens que pour les comptages
non_comptages = ["Nombre de lignes", "Nombre de colonnes", "Quantity min",
                 "Quantity max", "UnitPrice min", "UnitPrice max"]
tableau_initial.loc[tableau_initial["Contrôle"].isin(non_comptages), "% des lignes"] = None

# Affichage en vrai tableau
tableau_initial.style.format({"Avant": "{:,.2f}".format, "% des lignes": "{:.2f}"},
                             na_rep="").hide(axis="index") \
    .set_table_styles([{"selector": "th", "props": [("background-color", "#3b3f7a"),
                                                    ("color", "white"), ("text-align", "left")]}])

Contrôle,Avant,% des lignes
Nombre de lignes,"541,909.00",
Nombre de colonnes,8.00,
Doublons exacts (8 colonnes),"5,268.00",0.97
Quantity min,"-80,995.00",
Quantity max,"80,995.00",
UnitPrice min,"-11,062.06",
UnitPrice max,"38,970.00",
Quantity <= 0,"10,624.00",1.96
UnitPrice <= 0,"2,517.00",0.46
InvoiceNo commençant par C,"9,288.00",1.71


Un contrôle avant transformation est indispensable car il fournit l’état de référence de la source brute, qui ne peut plus être mesuré une fois les données modifiées.
Il permet de calculer les écarts avant/après, de vérifier qu’aucune ligne n’est perdue sans explication (réconciliation lignes source = doublons supprimés + lignes d’audit + lignes préparées), de prouver l’effet de chaque transformation et de détecter une erreur de préparation, comme la suppression involontaire de lignes à CustomerID manquant.

## **Analyse des anomalies**

In [5]:
n = len(df)

code = df["StockCode"].astype(str).str.strip()
inv = df["InvoiceNo"].astype(str)

# --- 1. Drapeaux (aucune suppression) ---
flags = pd.DataFrame(index=df.index)
flags["est_annulation"]        = inv.str.startswith("C")
flags["quantite_non_positive"] = df["Quantity"] <= 0
flags["prix_non_positif"]      = df["UnitPrice"] <= 0
flags["client_manquant"]       = df["CustomerID"].isna()
flags["description_manquante"] = df["Description"].isna()
flags["code_a_examiner"]       = ~code.str.fullmatch(r"\d{5}[A-Za-z]{0,2}")

# --- 2. Table d'anomalies ---
regles = [
    ("est_annulation",        "InvoiceNo commençant par C", "Annulation/retour = cas métier réel",
     "Exclu de FactVente MVP, conservé en audit"),
    ("quantite_non_positive", "Quantity <= 0",              "MVP : Quantity > 0",
     "Exclu de FactVente, conservé en audit"),
    ("prix_non_positif",      "UnitPrice <= 0",             "MVP : UnitPrice > 0",
     "Exclu de FactVente, conservé en audit"),
    ("client_manquant",       "CustomerID manquant",        "Vente réelle, client non identifié",
     "CONSERVÉ (rattachement ultérieur à « Client inconnu »)"),
    ("description_manquante", "Description manquante",      "Ne pas inventer de valeur",
     "Étiqueté à revoir, conservé"),
    ("code_a_examiner",       "StockCode hors format produit", "Classer via la référence (code + description + comportement)",
     "À classer (TypeLigne) ; A_VERIFIER si doute"),
]

table_anomalies = pd.DataFrame([
    {"Phénomène": ph,
     "Volume": int(flags[col].sum()),
     "% lignes": round(flags[col].mean() * 100, 2),
     "Justification": just,
     "Traitement": trait}
    for col, ph, just, trait in regles
])

# --- 3. Sauvegarde ---
table_anomalies.to_csv("table_anomalies.csv", index=False, encoding="utf-8-sig")

# --- 4. Affichage stylé (DERNIÈRE ligne de la cellule) ---
table_anomalies.style \
    .format({"Volume": "{:,.0f}", "% lignes": "{:.2f}"}, na_rep="") \
    .hide(axis="index") \
    .set_table_styles([{"selector": "th",
                        "props": [("background-color", "#3b3f7a"),
                                  ("color", "white"),
                                  ("text-align", "left")]}])

Phénomène,Volume,% lignes,Justification,Traitement
InvoiceNo commençant par C,"9,288",1.71,Annulation/retour = cas métier réel,"Exclu de FactVente MVP, conservé en audit"
Quantity <= 0,"10,624",1.96,MVP : Quantity > 0,"Exclu de FactVente, conservé en audit"
UnitPrice <= 0,"2,517",0.46,MVP : UnitPrice > 0,"Exclu de FactVente, conservé en audit"
CustomerID manquant,"135,080",24.93,"Vente réelle, client non identifié",CONSERVÉ (rattachement ultérieur à « Client inconnu »)
Description manquante,"1,454",0.27,Ne pas inventer de valeur,"Étiqueté à revoir, conservé"
StockCode hors format produit,"2,995",0.55,Classer via la référence (code + description + comportement),À classer (TypeLigne) ; A_VERIFIER si doute


**Une quantité négative est-elle nécessairement une erreur ? Quelle
règle du MVP s’applique ?**

Les quantités négatives sans « C » ne correspondent pas à des retours déclarés par un client. Elles semblent être des opérations internes (ajustements de stock, casse, pertes), ce qui est à confirmer par l’examen des descriptions. Elles sont exclues de FactVente (Quantity > 0) mais conservées en audit.

## **Normalisation sans invention**

In [6]:
import pandas as pd

df_prep = df.copy()   # on travaille sur une copie, la source brute reste intacte

colonnes_texte = ["Description", "Country", "StockCode"]
trace = []

for col in colonnes_texte:
    original = df_prep[col]

    # Trim : on ne touche qu'aux vraies chaînes (les NaN restent NaN)
    nettoye = original.where(original.isna(), original.astype(str).str.strip())

    df_prep[f"{col}_Trim"] = nettoye
    df_prep[f"{col}_Modifiee"] = original.notna() & (original != nettoye)

    trace.append({
        "Colonne": col,
        "Lignes modifiées": int(df_prep[f"{col}_Modifiee"].sum()),
        "% lignes": round(df_prep[f"{col}_Modifiee"].mean() * 100, 2),
        "Manquants avant": int(original.isna().sum()),
        "Manquants après": int(nettoye.isna().sum()),
    })

trace_transformations = pd.DataFrame(trace)
trace_transformations.style \
    .format({"Lignes modifiées": "{:,.0f}", "% lignes": "{:.2f}"}) \
    .hide(axis="index") \
    .set_table_styles([{"selector": "th",
                        "props": [("background-color", "#3b3f7a"),
                                  ("color", "white"),
                                  ("text-align", "left")]}])

Colonne,Lignes modifiées,% lignes,Manquants avant,Manquants après
Description,"113,453",20.94,1454,1454
Country,0,0.00,0,0
StockCode,"487,036",89.87,0,0


In [7]:
df_prep.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Description_Trim,Description_Modifiee,Country_Trim,Country_Modifiee,StockCode_Trim,StockCode_Modifiee
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,WHITE HANGING HEART T-LIGHT HOLDER,False,United Kingdom,False,85123A,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,WHITE METAL LANTERN,False,United Kingdom,False,71053,True
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,CREAM CUPID HEARTS COAT HANGER,False,United Kingdom,False,84406B,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,KNITTED UNION FLAG HOT WATER BOTTLE,False,United Kingdom,False,84029G,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,RED WOOLLY HOTTIE WHITE HEART.,False,United Kingdom,False,84029E,False


**Pourquoi conserver les valeurs originales lors d’une normalisation ?**

Pour la traçabilité : on peut prouver ce qui a été modifié, comparer avant/après, mesurer l’effet de la transformation et revenir à la source en cas d’erreur.
Une normalisation est une décision de préparation, pas une vérité : si elle s’avère mauvaise, l’original permet de la corriger sans recharger ni perdre d’information.

## **Dédoublement exacte**

In [8]:
import pandas as pd
from IPython.display import display

COLS_SOURCES = ["InvoiceNo", "StockCode", "Description", "Quantity",
                "InvoiceDate", "UnitPrice", "CustomerID", "Country"]

STYLE = [{"selector": "th", "props": [("background-color", "#3b3f7a"),
                                      ("color", "white"),
                                      ("text-align", "left")]}]

n_avant = len(df_prep)

# 1. Marquer (sans rien supprimer)
#    keep="first" : la 1re occurrence n'est PAS marquée, seules les copies en trop le sont
df_prep["est_doublon_exact"] = df_prep.duplicated(subset=COLS_SOURCES, keep="first")

# 2. Garder la trace des lignes qui vont être supprimées
df_doublons_supprimes = df_prep[df_prep["est_doublon_exact"]].copy()
df_doublons_supprimes.to_csv("doublons_supprimes.csv", index=False, encoding="utf-8-sig")

# 3. Supprimer uniquement les copies en trop
df_dedup = df_prep[~df_prep["est_doublon_exact"]].copy()

# 4. Compteur et vérification
n_supprimes = len(df_doublons_supprimes)
n_apres = len(df_dedup)

assert n_avant == n_apres + n_supprimes, "Bilan incohérent"
assert df_dedup.duplicated(subset=COLS_SOURCES).sum() == 0, "Il reste des doublons"

# 5. Tableau de trace
trace_dedup = pd.DataFrame({
    "Contrôle": ["Lignes avant", "Doublons exacts supprimés", "Lignes après",
                 "Doublons restants après"],
    "Valeur": [n_avant, n_supprimes, n_apres,
               int(df_dedup.duplicated(subset=COLS_SOURCES).sum())],
})

trace_dedup.style.format({"Valeur": "{:,.0f}"}).hide(axis="index").set_table_styles(STYLE)

Contrôle,Valeur
Lignes avant,"541,909"
Doublons exacts supprimés,"5,268"
Lignes après,"536,641"
Doublons restants après,0


**Pourquoi deux lignes ayant le même InvoiceNo et StockCode peuventelles être légitimes ?**

Deux lignes peuvent avoir le même InvoiceNo et le même StockCode tout en étant légitimes, car une facture peut contenir le même produit sur plusieurs lignes : saisies à des moments différents, quantités ajoutées séparément ou prix unitaires différents. 
Ces lignes diffèrent par Quantity, UnitPrice ou InvoiceDate. Seule l’identité sur les 8 colonnes sources prouve une copie technique. Dédoublonner sur (InvoiceNo + StockCode) seul supprimerait des ventes réelles et sous-estimerait le chiffre d’affaires.

## **Classification des lignes**

In [9]:
import pandas as pd

STYLE = [{"selector": "th", "props": [("background-color", "#3b3f7a"),
                                      ("color", "white"),
                                      ("text-align", "left")]}]

# ------------------------------------------------------------------
# À ADAPTER avec ta référence de classification (dossier dataset)
# ------------------------------------------------------------------
CODES_NON_COMMERCIAUX = {
    "POST", "DOT", "M", "D", "C2", "S", "B", "CRUK",
    "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2",
    "TEST001", "TEST002",
}
MOTS_NON_COMMERCIAUX = r"POSTAGE|MANUAL|DISCOUNT|BANK CHARGES|AMAZON FEE|ADJUST|CARRIAGE"
FORMAT_PRODUIT = r"\d{5}[A-Z]{0,2}"      # ex. 85123, 85123A

# ------------------------------------------------------------------
# Données d'entrée : source dédoublonnée + colonnes Trim (étapes 03-04)
# ------------------------------------------------------------------
d = df_dedup.copy()
code = d["StockCode_Trim"].astype(str).str.upper()
desc = d["Description_Trim"].fillna("").str.upper()

est_annulation = d["InvoiceNo"].astype(str).str.startswith("C")

# Indices (un par critère)
code_non_comm   = code.isin(CODES_NON_COMMERCIAUX)
code_format_ok  = code.str.fullmatch(FORMAT_PRODUIT)
desc_manquante  = d["Description_Trim"].isna() | (desc == "")
desc_suspecte   = desc.str.contains(MOTS_NON_COMMERCIAUX, regex=True)

# ------------------------------------------------------------------
# TypeLigne + motif de classification
# ------------------------------------------------------------------
d["TypeLigne"] = "A_VERIFIER"
d["MotifClassification"] = "Code hors format produit et hors liste non commerciale"

# 1) Code connu comme non commercial
m = code_non_comm
d.loc[m, "TypeLigne"] = "NON_COMMERCIAL"
d.loc[m, "MotifClassification"] = "StockCode dans la liste non commerciale"

# 2) Code au format produit
m = code_format_ok & ~code_non_comm
d.loc[m, "TypeLigne"] = "PRODUIT_COMMERCIAL"
d.loc[m, "MotifClassification"] = "Code format produit + description cohérente"

# 3) Format produit mais description absente ou suspecte : on ne tranche pas
m = code_format_ok & ~code_non_comm & desc_manquante
d.loc[m, "TypeLigne"] = "A_VERIFIER"
d.loc[m, "MotifClassification"] = "Code format produit mais description manquante"

m = code_format_ok & ~code_non_comm & ~desc_manquante & desc_suspecte
d.loc[m, "TypeLigne"] = "A_VERIFIER"
d.loc[m, "MotifClassification"] = "Code format produit mais description de type frais/ajustement"

# ------------------------------------------------------------------
# InclureFactVente : les 4 règles du périmètre MVP
# ------------------------------------------------------------------
d["InclureFactVente"] = (
    ~est_annulation
    & (d["Quantity"] > 0)
    & (d["UnitPrice"] > 0)
    & (d["TypeLigne"] == "PRODUIT_COMMERCIAL")
)

# Motif d'exclusion (pour la couche d'audit), premier motif rencontré
d["MotifExclusion"] = ""
d.loc[~d["InclureFactVente"] & (d["TypeLigne"] == "A_VERIFIER"),     "MotifExclusion"] = "Type A_VERIFIER"
d.loc[~d["InclureFactVente"] & (d["TypeLigne"] == "NON_COMMERCIAL"), "MotifExclusion"] = "Ligne non commerciale"
d.loc[~d["InclureFactVente"] & (d["UnitPrice"] <= 0),                "MotifExclusion"] = "UnitPrice <= 0"
d.loc[~d["InclureFactVente"] & (d["Quantity"] <= 0),                 "MotifExclusion"] = "Quantity <= 0"
d.loc[~d["InclureFactVente"] & est_annulation,                       "MotifExclusion"] = "Annulation (InvoiceNo C)"

df_classe = d   # table de classification (aucune ligne supprimée)

# ------------------------------------------------------------------
# Contrôles automatiques
# ------------------------------------------------------------------
assert len(df_classe) == len(df_dedup), "Des lignes ont disparu"
assert not (df_classe["InclureFactVente"] & (df_classe["TypeLigne"] != "PRODUIT_COMMERCIAL")).any(), \
    "Une ligne non commerciale est dans le périmètre de vente"
assert not (df_classe["InclureFactVente"] & df_classe["InvoiceNo"].astype(str).str.startswith("C")).any()
print("Contrôles OK : aucune ligne perdue, aucune ligne non commerciale dans le périmètre")

# ------------------------------------------------------------------
# Tableau de classification (dernière ligne = affichage stylé)
# ------------------------------------------------------------------
table_classif = (df_classe.groupby("TypeLigne")
                 .agg(Lignes=("TypeLigne", "size"),
                      Incluses_FactVente=("InclureFactVente", "sum"))
                 .reset_index())
table_classif["Exclues"] = table_classif["Lignes"] - table_classif["Incluses_FactVente"]
table_classif["% lignes"] = (table_classif["Lignes"] / len(df_classe) * 100).round(2)

table_classif.style.format({"Lignes": "{:,.0f}", "Incluses_FactVente": "{:,.0f}",
                            "Exclues": "{:,.0f}", "% lignes": "{:.2f}"}) \
    .hide(axis="index").set_table_styles(STYLE)

Contrôles OK : aucune ligne perdue, aucune ligne non commerciale dans le périmètre


TypeLigne,Lignes,Incluses_FactVente,Exclues,% lignes
A_VERIFIER,"1,731",0,"1,731",0.32
NON_COMMERCIAL,"2,906",0,"2,906",0.54
PRODUIT_COMMERCIAL,"532,004","522,331","9,673",99.14


In [10]:
df_classe[df_classe["TypeLigne"] != "PRODUIT_COMMERCIAL"]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Description_Trim,Description_Modifiee,Country_Trim,Country_Modifiee,StockCode_Trim,StockCode_Modifiee,est_doublon_exact,TypeLigne,MotifClassification,InclureFactVente,MotifExclusion
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France,POSTAGE,False,France,False,POST,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,Discount,False,United Kingdom,False,D,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Annulation (InvoiceNo C)
386,536403,POST,POSTAGE,1,2010-12-01 11:27:00,15.00,12791.0,Netherlands,POSTAGE,False,Netherlands,False,POST,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom,NaN,False,United Kingdom,False,22139,True,False,A_VERIFIER,Code format produit mais description manquante,False,UnitPrice <= 0
1123,536527,POST,POSTAGE,1,2010-12-01 13:04:00,18.00,12662.0,Germany,POSTAGE,False,Germany,False,POST,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541540,581498,DOT,DOTCOM POSTAGE,1,2011-12-09 10:26:00,1714.17,NaN,United Kingdom,DOTCOM POSTAGE,False,United Kingdom,False,DOT,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,Manual,False,United Kingdom,False,M,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Annulation (InvoiceNo C)
541730,581570,POST,POSTAGE,1,2011-12-09 11:59:00,18.00,12662.0,Germany,POSTAGE,False,Germany,False,POST,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale
541767,581574,POST,POSTAGE,2,2011-12-09 12:09:00,18.00,12526.0,Germany,POSTAGE,False,Germany,False,POST,False,False,NON_COMMERCIAL,StockCode dans la liste non commerciale,False,Ligne non commerciale


La présence d’une lettre dans le StockCode ne suffit pas à exclure une ligne, car de nombreux produits réels ont un code alphanumérique (par exemple 85123A, une variante de produit). Exclure toute ligne contenant une lettre supprimerait des ventes réelles. À l’inverse, certains codes non commerciaux (POST, M, D) sont de courts codes composés de lettres. La classification croise donc StockCode, Description et comportement (Quantity, UnitPrice), selon la référence validée, et place les cas douteux en A_VERIFIER.

## **Construction de la source préparée**

### **ETAPE 1: Comptage de lignes avant/après chaque filtre**

Application de périmètre validé du MVP

In [11]:
STYLE = [{"selector": "th", "props": [("background-color", "#3b3f7a"),
                                      ("color", "white"),
                                      ("text-align", "left")]}]

est_annulation = df_classe["InvoiceNo"].astype(str).str.startswith("C")

filtres = [
    ("InvoiceNo non annulé",        ~est_annulation),
    ("Quantity > 0",                df_classe["Quantity"] > 0),
    ("UnitPrice > 0",               df_classe["UnitPrice"] > 0),
    ("TypeLigne = PRODUIT_COMMERCIAL", df_classe["TypeLigne"] == "PRODUIT_COMMERCIAL"),
]

masque_cumule = pd.Series(True, index=df_classe.index)
etapes = [("Source dédoublonnée (départ)", len(df_classe), 0)]
avant = len(df_classe)

for nom, cond in filtres:
    masque_cumule = masque_cumule & cond
    apres = int(masque_cumule.sum())
    etapes.append((f"Après filtre : {nom}", apres, avant - apres))
    avant = apres

table_filtres = pd.DataFrame(etapes, columns=["Étape", "Lignes restantes", "Lignes exclues par ce filtre"])

# Cohérence avec InclureFactVente de l'étape 05
assert masque_cumule.equals(df_classe["InclureFactVente"]), "Le périmètre diffère d'InclureFactVente"

table_filtres.style.format({"Lignes restantes": "{:,.0f}", "Lignes exclues par ce filtre": "{:,.0f}"}) \
    .hide(axis="index").set_table_styles(STYLE)

Étape,Lignes restantes,Lignes exclues par ce filtre
Source dédoublonnée (départ),"536,641",0
Après filtre : InvoiceNo non annulé,"527,390","9,251"
Après filtre : Quantity > 0,"526,054","1,336"
Après filtre : UnitPrice > 0,"524,878","1,176"
Après filtre : TypeLigne = PRODUIT_COMMERCIAL,"522,331","2,547"


### **ETAPE 2 : Séparation L2 / audit**


>**df_l2** contient les lignes incluse dans factvente
>
> **df_audit** contient les lignes qui ne repond pas aux perimetre de MVP 


In [12]:
# L2 : ventes du périmètre MVP
df_l2 = df_classe[df_classe["InclureFactVente"]].copy()

# Audit : lignes exclues, avec leur motif
df_audit = df_classe[~df_classe["InclureFactVente"]].copy()

# Réconciliation : aucune ligne perdue
assert len(df_l2) + len(df_audit) == len(df_classe), "Des lignes ont disparu"
assert (df_audit["MotifExclusion"] != "").all(), "Une ligne d'audit n'a pas de motif"
assert not df_l2["InvoiceNo"].astype(str).str.startswith("C").any()
assert (df_l2["Quantity"] > 0).all() and (df_l2["UnitPrice"] > 0).all()
assert (df_l2["TypeLigne"] == "PRODUIT_COMMERCIAL").all()

# Bilan complet depuis la source brute
assert len(df) == n_supprimes + len(df_audit) + len(df_l2), "Bilan source incohérent"
print("Contrôles OK")

Contrôles OK


### **ETAPE 3 : Bilan de réconciliation**

In [13]:
bilan = pd.DataFrame({
    "Destination": ["Source brute", "Doublons exacts supprimés",
                    "Lignes d'audit (exclues de FactVente)", "Lignes L2 (préparées)", "Total contrôlé"],
    "Lignes": [len(df), n_supprimes, len(df_audit), len(df_l2),
               n_supprimes + len(df_audit) + len(df_l2)],
})
bilan["% de la source"] = (bilan["Lignes"] / len(df) * 100).round(2)

bilan.style.format({"Lignes": "{:,.0f}", "% de la source": "{:.2f}"}) \
    .hide(axis="index").set_table_styles(STYLE)

Destination,Lignes,% de la source
Source brute,"541,909",100.00
Doublons exacts supprimés,"5,268",0.97
Lignes d'audit (exclues de FactVente),"14,310",2.64
Lignes L2 (préparées),"522,331",96.39
Total contrôlé,"541,909",100.00


### **ETAPE 4 : Motifs d’exclusion**

In [14]:
motifs = (df_audit["MotifExclusion"].value_counts()
          .rename_axis("MotifExclusion").reset_index(name="Lignes"))

motifs.style.format({"Lignes": "{:,.0f}"}).hide(axis="index").set_table_styles(STYLE)

MotifExclusion,Lignes
Annulation (InvoiceNo C),"9,251"
Ligne non commerciale,"2,307"
Quantity <= 0,"1,336"
UnitPrice <= 0,"1,176"
Type A_VERIFIER,240


### **ETAPE 5 : Sauvegarde des deux couches**


> **df_l2** : données préparées respectant le périmètre FactVente MVP (facture non annulée, Quantity > 0, UnitPrice > 0, PRODUIT_COMMERCIAL). Les lignes sans CustomerID y sont conservées.
> 
> **df_audit** : lignes exclues de FactVente mais conservées, avec leur motif (annulations/retours, quantités ou prix non positifs, lignes non commerciales, cas A_VERIFIER).
> 
> **df_doublons_supprimes** : copies exactes en trop retirées de la source (seule suppression définitive), tracées pour la réconciliation.


In [15]:
df_l2.to_csv("L2_donnees_preparees.csv", index=False, encoding="utf-8-sig")
df_audit.to_csv("L2_audit_lignes_exclues.csv", index=False, encoding="utf-8-sig")
df_doublons_supprimes.to_csv("L2_doublons_supprimes.csv", index=False, encoding="utf-8-sig")

Ces trois tableaux forment la réconciliation complète :

> lignes source brute = **df_doublons_supprimes** + **df_audit** + **df_l2**

**Quelle différence entre « exclue de FactVente » et « supprimée de la source » ?**

* Une ligne ***exclue*** de FactVente reste dans la couche d’audit, avec son motif d’exclusion : elle est tracée, récupérable et analysable (par exemple les retours).
  
* Une ligne ***supprimée*** de la source n’existe plus nulle part. Dans ce TP, seuls les doublons exacts sont supprimés (copies techniques, tracées par un compteur). Annulations, quantités ou prix non positifs et lignes non commerciales sont exclues du périmètre MVP mais conservées en audit, ce qui permet la réconciliation : source = doublons + audit + L2.

## **Création de la colonne dérivé : "Montantligne"**

**Principe**

MontantLigne = Quantity × UnitPrice, créé uniquement sur df_l2 (la couche préparée).

#### 1. **creation de la colonne**

In [16]:
df_l2["MontantLigne"] = (df_l2["Quantity"] * df_l2["UnitPrice"]).round(2)

#### 2. **contrôles automatiques**

In [17]:
# 1. La colonne existe et Quantity / UnitPrice sont toujours là
assert {"Quantity", "UnitPrice", "MontantLigne"}.issubset(df_l2.columns)

# 2. Aucun montant manquant
assert df_l2["MontantLigne"].notna().all(), "MontantLigne contient des valeurs manquantes"

# 3. Cohérence : tous les montants sont positifs (puisque Quantity > 0 et UnitPrice > 0)
assert (df_l2["MontantLigne"] > 0).all(), "Un montant n'est pas positif"

# 4. Recalcul indépendant de la somme
somme_recalculee = (df_l2["Quantity"] * df_l2["UnitPrice"]).sum()
assert abs(df_l2["MontantLigne"].sum() - somme_recalculee) < 1, "Écart sur la somme"

print("Contrôles OK : MontantLigne cohérent")

Contrôles OK : MontantLigne cohérent


In [18]:
len(df_audit)

14310


**Pourquoi conserver les deux composantes alors que MontantLigneexiste ?**

***Quantity et UnitPrice*** sont conservés car ***MontantLigne*** est une colonne dérivée (Quantity × UnitPrice) : les données d’origine garantissent la traçabilité et permettent de recalculer et contrôler le montant.
Elles permettent aussi d’analyser séparément le volume vendu et le prix *(prix moyen, évolution des prix)*, ce qu’un montant seul ne permet pas. 


## **Controles techniques :**

**Principe**



> * **Avant** = la source brute df (541 909 lignes).
> * **Après** = la couche préparée df_l2.
> * **Écart** = Après − Avant.
> * **Interprétation** = une phrase qui explique l’écart par une règle.



In [19]:
import pandas as pd

STYLE = [{"selector": "th", "props": [("background-color", "#3b3f7a"),
                                      ("color", "white"),
                                      ("text-align", "left")]}]

def est_c(s):
    return s.astype(str).str.startswith("C")

# Classification "avant" : on applique la même logique que l'étape 05 à la source brute
# df_classe porte TypeLigne mais seulement sur la source dédoublonnée.
# Pour comparer à périmètre égal, on mesure "avant" sur df_classe (dédoublonnée),
# et on garde le nombre de lignes / doublons de la source brute.
non_comm_avant = int((df_classe["TypeLigne"] != "PRODUIT_COMMERCIAL").sum())
non_comm_apres = int((df_l2["TypeLigne"] != "PRODUIT_COMMERCIAL").sum())

lignes = [
    ("Nombre de lignes",          len(df),
                                  len(df_l2),
     "Doublons retirés, puis lignes exclues vers l'audit (voir bilan)"),
    ("Doublons exacts",           int(df.duplicated(subset=COLS_SOURCES).sum()),
                                  int(df_l2.duplicated(subset=COLS_SOURCES).sum()),
     "Seule suppression définitive : copies exactes sur les 8 colonnes"),
    ("CustomerID manquants",      int(df["CustomerID"].isna().sum()),
                                  int(df_l2["CustomerID"].isna().sum()),
     "Conservés (Client inconnu) ; la baisse vient uniquement des autres filtres"),
    ("Lignes annulées / retours", int(est_c(df["InvoiceNo"]).sum()),
                                  int(est_c(df_l2["InvoiceNo"]).sum()),
     "Exclues de FactVente MVP, conservées en audit"),
    ("Lignes UnitPrice <= 0",     int((df["UnitPrice"] <= 0).sum()),
                                  int((df_l2["UnitPrice"] <= 0).sum()),
     "Règle MVP UnitPrice > 0 ; conservées en audit"),
    ("Lignes Quantity <= 0",      int((df["Quantity"] <= 0).sum()),
                                  int((df_l2["Quantity"] <= 0).sum()),
     "Règle MVP Quantity > 0 ; conservées en audit"),
    ("Lignes non commerciales",   non_comm_avant,
                                  non_comm_apres,
     "Hors PRODUIT_COMMERCIAL (NON_COMMERCIAL + A_VERIFIER) ; avant mesuré après dédoublonnage"),
    ("Somme MontantLigne",        round((df["Quantity"] * df["UnitPrice"]).sum(), 2),
                                  round(df_l2["MontantLigne"].sum(), 2),
     "Retrait des doublons, annulations, prix/quantités non positifs et lignes non commerciales"),
]

table_controles = pd.DataFrame(lignes, columns=["Contrôle", "Avant", "Après", "Interprétation"])
table_controles.insert(3, "Écart", table_controles["Après"] - table_controles["Avant"])
table_controles = table_controles[["Contrôle", "Avant", "Après", "Écart", "Interprétation"]]
table_controles.style \
    .format({"Avant": "{:,.2f}", "Après": "{:,.2f}", "Écart": "{:,.2f}"}) \
    .hide(axis="index") \
    .set_table_styles(STYLE + [{"selector": "td", "props": [("text-align", "left")]}])

Contrôle,Avant,Après,Écart,Interprétation
Nombre de lignes,"541,909.00","522,331.00","-19,578.00","Doublons retirés, puis lignes exclues vers l'audit (voir bilan)"
Doublons exacts,"5,268.00",0.00,"-5,268.00",Seule suppression définitive : copies exactes sur les 8 colonnes
CustomerID manquants,"135,080.00","131,322.00","-3,758.00",Conservés (Client inconnu) ; la baisse vient uniquement des autres filtres
Lignes annulées / retours,"9,288.00",0.00,"-9,288.00","Exclues de FactVente MVP, conservées en audit"
Lignes UnitPrice <= 0,"2,517.00",0.00,"-2,517.00",Règle MVP UnitPrice > 0 ; conservées en audit
Lignes Quantity <= 0,"10,624.00",0.00,"-10,624.00",Règle MVP Quantity > 0 ; conservées en audit
Lignes non commerciales,"4,637.00",0.00,"-4,637.00",Hors PRODUIT_COMMERCIAL (NON_COMMERCIAL + A_VERIFIER) ; avant mesuré après dédoublonnage
Somme MontantLigne,"9,747,747.93","10,244,262.73","496,514.80","Retrait des doublons, annulations, prix/quantités non positifs et lignes non commerciales"


## **Analyse et intérprétation**

**OBSERVATION:** **quels changements sont mesurés ?**
> La source brute contient 541 909 lignes.
> 
> Après préparation, L2 contient **522,331.00** lignes.
> 
> Le bilan se décompose ainsi :
> 
> * **5 268** doublons exacts supprimés,
> *  **14310** lignes exclues vers l’audit et
> *  **522,331.00** lignes conservées dans L2, soit un total de 541 909.
> * Les doublons exacts passent de **5 268 à 0**.
> * Les lignes annulées, Quantity ≤ 0, UnitPrice ≤ 0 et non commerciales **passent à 0 dans L2**.
> * Les CustomerID manquants passent de 135 080 à **131,322.00** .
> * La somme de MontantLigne passe de 9,747,747.93 à **10,244,262.73**.

**ANALYSE : pourquoi ces changements apparaissent-ils ?**

>* **Nombre de lignes** : baisse due d’abord aux doublons retirés, puis aux filtres du périmètre MVP.
>* **Doublons** : les copies exactes sur les huit colonnes ont été retirées, une occurrence conservée.
>* **Annulations, Quantity ≤ 0, UnitPrice ≤ 0, non commerciales** : ces lignes ne respectent pas au moins un des quatre filtres, donc elles sont déplacées vers l’audit.
>* **CustomerID manquants** : la baisse ne vient pas d’un filtre sur CustomerID (il n’y en a aucun). Elle vient des autres filtres, car une partie de ces lignes sont aussi des annulations, des prix ≤ 0 ou des codes non commerciaux.
>* **Somme MontantLigne** : l’écart s’explique par les doublons retirés et par les lignes exclues (les annulations, qui ont un montant négatif, font en général augmenter la somme quand on les retire).

**INTERPRÉTATION : quelles règles métier expliquent les exclusions/conservations ?**

>* **Doublons exacts supprimés**: copies techniques, qui compteraient une vente plusieurs fois.
>* **Annulations/retours** : événements métier réels, hors périmètre car le CA du MVP est un CA de ventes hors >annulations/retours. Conservés en audit.
>* **Quantity ≤ 0 et UnitPrice ≤ 0** : règle MVP Quantity > 0 et UnitPrice > 0. Une quantité négative n’est pas forcément une >erreur. Conservées en audit.
>* **Codes non commerciaux et A_VERIFIER** : seules les lignes PRODUIT_COMMERCIAL sont des ventes de produits. Les cas douteux sont exclus mais documentés.
>* **CustomerID manquant**: ventes réelles conservées pour rattachement ultérieur à « Client inconnu ». Aucune valeur inventée.

**DÉCISION : quelle version de L2 transmettre à TP3 et quelles traces garder ?**

>**Version transmise** : df_l2, qui contient les ventes du périmètre MVP (facture non annulée, Quantity > 0, UnitPrice > 0, PRODUIT_COMMERCIAL), avec les lignes à CustomerID manquant conservées et la colonne dérivée MontantLigne. Aucune clé de substitution ni dimension n’est créée (réservé à TP3).

>**Traces conservées** :
>
>* ***L2_donnees_preparees.csv*** : la couche préparée,
>* ***L2_audit_lignes_exclues.csv*** : les lignes exclues avec leur MotifExclusion,
>* ***L2_doublons_supprimes.csv*** : les copies supprimées,
le script/notebook reproductible,
le tableau avant/après et le bilan de réconciliation : source = doublons + audit + L2,
la liste des cas A_VERIFIER documentés.